# Promoter Activity Functions (CAPÍTULO 5.1)

In [ ]:
import matplotlib.pyplot as plt
import scienceplots
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import pandas as pd
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from numbalsoda import lsoda_sig, lsoda
from numba import types

plt.style.use('science')
width = 'thesis'


def set_size(width, fraction=1, subplots=(1, 1)):
    """Set figure dimensions to avoid scaling in LaTeX.

    Parameters
    ----------
    width: float or string
            Document width in points, or string of predined document type
    fraction: float, optional
            Fraction of the width which you wish the figure to occupy
    subplots: array-like, optional
            The number of rows and columns of subplots.
    Returns
    -------
    fig_dim: tuple
            Dimensions of figure in inches
    """
    if width == 'thesis':
        width_pt = 426.79135
    elif width == 'beamer':
        width_pt = 307.28987
    else:
        width_pt = width

    # Width of figure (in pts)
    fig_width_pt = width_pt * fraction
    # Convert from pt to inches
    inches_per_pt = 1 / 72.27

    # Golden ratio to set aesthetic figure height
    # https://disq.us/p/2940ij3
    golden_ratio = (5**.5 - 1) / 2

    # Figure width in inches
    fig_width_in = fig_width_pt * inches_per_pt
    # Figure height in inches
    fig_height_in = fig_width_in * golden_ratio * (subplots[0] / subplots[1])

    return (fig_width_in, fig_height_in)

## 1.a Promoter Activity - Thermodynamic Model

In [ ]:
# First we have to define the thermodynamic equations for the promoter
@nb.njit(fastmath=True)
def phiProm(A, R, Ka, Kr, w_AR, w_R, beta):

    return ((Ka * A) / (1.0 +
                        Ka * A +
                        Ka * Kr * A * R * w_AR +
                        Kr * R * w_R))

# Define a space for both activators and repressors
c1 = np.linspace(1e-5, 1e-4, 100)
c2 = np.linspace(1e-4, 1e-3, 100)
c3 = np.linspace(1e-3, 1e-2, 100)
c4 = np.linspace(1e-2, 1e-1, 100)
c5 = np.linspace(1e-1, 1e0, 100)
A_space = np.concatenate((c1, c2, c3, c4, c5))
A_space = np.unique(A_space)
R_space = np.concatenate((c1, c2, c3, c4, c5))
R_space = np.unique(R_space)

# Create a meshgrid for both spaces
AA, RR = np.meshgrid(A_space, R_space)

# Define the parameters of each promoter
Ka_LUX = 10.0
Kr_TET = 1000.0
Ka_LAS = 10.0
Kr_LAC = 10.0
w_LT =  1e-5
w_LL = 1e-5
w_T = 2.0
w_L = 4.0

# Evaluate the promoters over the space (considering both the pLT and pLL promoters)
pLT = phiProm(AA, RR, Ka_LUX, Kr_TET, w_LT, w_T)
pLL = phiProm(AA, RR, Ka_LAS, Kr_LAC, w_LL, w_L)

# Create a plotly interactive 3D plot, using sliders to change the parameters
fig = make_subplots(rows=1, cols=1, specs=[[{"type": "surface"}]])

# Add the surface plot
fig.add_trace(go.Surface(x=np.log10(AA), y=np.log10(RR), z=pLT, colorscale="Viridis", opacity=0.75), row=1, col=1)
fig.add_trace(go.Surface(x=np.log10(AA), y=np.log10(RR), z=pLL, colorscale="Inferno", colorbar_x=1.15, opacity=0.75), row=1, col=1)

# Show the plot
fig.update_layout(height=800, 
                  width=1000,
                  title="Promoter Activity vs Activators and Repressors")
fig.update_scenes(xaxis_title_text="Activator (uM)",  
                  yaxis_title_text="Repressor (uM)",  
                  zaxis_title_text="Activity")
fig.show()

In [ ]:
# Now repeat the same 3D graph as above but using matplotlib instead of plotly
fig = plt.figure(figsize=set_size(width, fraction=1, subplots=(2, 1)), dpi=300)
ax = fig.add_subplot(211,projection='3d')
ax2 = fig.add_subplot(212,projection='3d')

# Plot the surfaces
surf1 = ax.plot_surface(np.log10(AA), np.log10(RR), pLT, cmap=plt.cm.viridis, linewidth=0, antialiased=True, alpha=1.0)
surf2 = ax2.plot_surface(np.log10(AA), np.log10(RR), pLL, cmap=plt.cm.inferno, linewidth=0, antialiased=True, alpha=1.0)

# Add a colorbar for each surface and a label for them
fig.colorbar(surf1, ax=ax, shrink=0.5, aspect=10)
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10)

# Set the labels
ax.set_xlabel('$\log_{10}([A])$ ($\mu$M)')
ax.set_ylabel('$\log_{10}([R])$ ($\mu$M)')
ax.set_zlabel('Actividad (U.A.)')
ax.set_title('Respuesta del promotor pLux76Tet\nfrente a un gradiente de activadores y represores')

# Set the labels for the second plot
ax2.set_xlabel('$\log_{10}([A])$ ($\mu$M)')
ax2.set_ylabel('$\log_{10}([R])$ ($\mu$M)')
ax.set_zlabel('Actividad (U.A.)')
ax2.set_title('Respuesta del promotor pLas81Lac\nfrente a un gradiente de activadores y represores')

# Tight layout
plt.tight_layout()

# Change the view
ax.view_init(30, 45)
ax2.view_init(30, 45)

# Save the figure
plt.savefig('promoter_activity_thesis_SHERMAN.pdf', dpi=600, bbox_inches='tight')

## 1.b Promoter Activity - Hill Function

In [ ]:
# From Weaver and Buceta
@nb.njit(fastmath=True)
def toggleSwitch(A, R, Kr, Ka, n_REP, n_ACT):

    return (((A/Ka)**n_ACT)/(1 + (A/Ka)**n_ACT + (R/Kr)**n_REP))

# Define a space for both activators and repressors
c1 = np.linspace(1e-5, 1e-4, 100)
c2 = np.linspace(1e-4, 1e-3, 100)
c3 = np.linspace(1e-3, 1e-2, 100)
c4 = np.linspace(1e-2, 1e-1, 100)
c5 = np.linspace(1e-1, 1e0, 100)
A_space = np.concatenate((c1, c2, c3, c4, c5))
A_space = np.unique(A_space)
R_space = np.concatenate((c1, c2, c3, c4, c5))
R_space = np.unique(R_space)

# Create a meshgrid for both spaces
AA, RR = np.meshgrid(A_space, R_space)

# Define the parameters of each promoter
Ka_LUX = 0.1
Kr_TET = 0.001
Ka_LAS = 0.1
Kr_LAC = 0.1
n_TET = 2 
n_LAC = 4 
n_LUX = 2
n_LAS = 2

# Evaluate the promoters over the space (considering both the pLT and pLL promoters)
pLT = toggleSwitch(AA, RR, Kr_TET, Ka_LUX, n_TET, n_LUX)
pLL = toggleSwitch(AA, RR, Kr_LAC, Ka_LAS, n_LAC, n_LAS)

# Create a plotly interactive 3D plot, using sliders to change the parameters
fig = make_subplots(rows=1, cols=1, specs=[[{"type": "surface"}]])

# Add the surface plot
fig.add_trace(go.Surface(x=np.log10(AA), y=np.log10(RR), z=pLT, colorscale="Viridis", opacity=0.75), row=1, col=1)
fig.add_trace(go.Surface(x=np.log10(AA), y=np.log10(RR), z=pLL, colorscale="Inferno", colorbar_x=1.15, opacity=0.75), row=1, col=1)

# Change the y and x axis to log scale
fig.update_xaxes(type="log", title_text="Activator (uM)", row=1, col=1)
fig.update_yaxes(type="log", title_text="Repressor (uM)", row=1, col=1)

# Show the plot
fig.update_layout(height=800, 
                  width=1000,
                  title="Promoter Activity vs Activators and Repressors",
                  scene=dict(
                    xaxis_title='Activator (uM)',
                    yaxis_title='Repressor (uM)',
                    zaxis_title='Promoter Activity'
                  ))
fig.show()

In [ ]:
# Now repeat the same 3D graph as above but using matplotlib instead of plotly
# fig = plt.figure(figsize=(10, 5), dpi=300)
fig = plt.figure(figsize=set_size(width, fraction=1, subplots=(2, 1)), dpi=300)
ax = fig.add_subplot(211,projection='3d')
ax2 = fig.add_subplot(212,projection='3d')

# Plot the surfaces
surf1 = ax.plot_surface(np.log10(AA), np.log10(RR), pLT, cmap=plt.cm.viridis, linewidth=0, antialiased=True, alpha=1.0)
surf2 = ax2.plot_surface(np.log10(AA), np.log10(RR), pLL, cmap=plt.cm.inferno, linewidth=0, antialiased=True, alpha=1.0)

# Add a colorbar for each surface and a label for them
fig.colorbar(surf1, ax=ax, shrink=0.5, aspect=10)
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10)

# Set the labels
ax.set_xlabel('$\log_{10}([A])$ ($\mu$M)')
ax.set_ylabel('$\log_{10}([R])$ ($\mu$M)')
ax.set_zlabel('Actividad (U.A.)')
ax.set_title('Respuesta del promotor pLux76Tet\nfrente a un gradiente de activadores y represores')

# Set the labels for the second plot
ax2.set_xlabel('$\log_{10}([A])$ ($\mu$M)')
ax2.set_ylabel('$\log_{10}([R])$ ($\mu$M)')
ax.set_zlabel('Actividad (U.A.)')
ax2.set_title('Respuesta del promotor pLas81Lac\nfrente a un gradiente de activadores y represores')

# Tight layout
plt.tight_layout()

# Change the view
ax.view_init(30, 45)
ax2.view_init(30, 45)

# Save the figure
plt.savefig('promoter_activity_thesis_WEAVER_BUCETA.pdf', dpi=600, bbox_inches='tight')

## 1.c Promoter Activity - Simplified Function

In [ ]:
@nb.jit(nopython=True, fastmath=True)
def hillLikeFunction(HSL, K, n):
    """
    This function returns the value of a hill-like function
    Inputs:
        - K: Association constant
        - HSL: HSL concentration
        - n: Hill coefficient
    Returns:
        - The value of the hill-like function
    """
    return ((K * HSL)/(1 + K * HSL)) ** n


@nb.jit(nopython=True, fastmath=True)
def promoterFunction(HSL, REP, K_A, K_1A, K_REP, n_A, n_REP, c):
    """
    This function returns the value of the promoter function
    Inputs:
        - HSL1: Concentration of HSL1
        - REP: Concentration of the repressor
        - b_0: Basal transcription rate for the activated promoter
        - K_A: Association constant of the activator
        - K_1A: Association constant of the activator to HSL
        - K_REP: Association constant of the repressor to the promoter
        - n_A: Hill coefficient of the activator
        - c: Tx/Tl constant (assuming QSS)
    Returns:
        - The value of the promoter function
    """
    return (K_A * (1/0.02) * c * hillLikeFunction(HSL, K_1A, n_A)) / (
                1
                + K_A * (1/0.02) * c * hillLikeFunction(HSL
                , K_1A, n_A)
                + (K_REP*REP)**n_REP)
            #) * (1 / (1 + (K_REP*REP)**n_REP))



# Define a space for both activators and repressors
c1 = np.linspace(1e-5, 1e-4, 100)
c2 = np.linspace(1e-4, 1e-3, 100)
c3 = np.linspace(1e-3, 1e-2, 100)
c4 = np.linspace(1e-2, 1e-1, 100)
c5 = np.linspace(1e-1, 1e0, 100)
c6 = np.linspace(1e0, 1e1, 100)
A_space = np.concatenate((c1, c2, c3, c4, c5, c6))
A_space = np.unique(A_space)
R_space = np.concatenate((c1, c2, c3, c4, c5, c6))
R_space = np.unique(R_space)

# Create a meshgrid for both spaces
AA, RR = np.meshgrid(A_space, R_space)

# Define the parameters of each promoter
# ----------------------------------------------------------------------------------------------------------------------
# Parameter Definitions
# ----------------------------------------------------------------------------------------------------------------------
# Transcription and translation parameters
k_tx = 1e-1
k_tr = 5e0
d_mrna = 5e-1
d_p = 5e-2

# LuxR / LasR and C6i / C12i parameters
K_1R = 1/0.10  # uM^-1
K_1S = 1/0.10  # uM^-1
K_GR = 1/0.015  # uM^-2
K_GS = 1/0.015 # uM^-2

# Promoter parameters
K_TET = 1/0.050  # uM^-1
K_LAC = 1/0.500 # uM^-1
n_R = 2.0
n_S = 2.0
n_REPTET = 2.0
n_REPLAC = 4.0
b_0 = 0.0050  # No units
b_1 = 0.0250  # No units

# Calculate the c constant
c = ((k_tx * k_tr) / (d_mrna * d_p)) ** 2

# Evaluate the promoters over the space (considering both the pLT and pLL promoters)
pLT = promoterFunction(AA, RR, K_TET, K_1R, K_GR, n_R, n_REPTET, c)
pLL = promoterFunction(AA, RR, K_LAC, K_1S, K_GS, n_S, n_REPLAC, c)

# Create a plotly interactive 3D plot, using sliders to change the parameters
fig = make_subplots(rows=1, cols=1, specs=[[{"type": "surface"}]])

# Add the surface plot
fig.add_trace(go.Surface(x=np.log10(AA), y=np.log10(RR), z=pLT, colorscale="Viridis", opacity=0.75), row=1, col=1)
fig.add_trace(go.Surface(x=np.log10(AA), y=np.log10(RR), z=pLL, colorscale="Inferno", colorbar_x=1.15, opacity=0.75), row=1, col=1)

# Show the plot
fig.update_layout(height=800, 
                  width=1000,
                  title="Promoter Activity vs Activators and Repressors")
fig.update_scenes(xaxis_title_text="Activator (uM)",  
                  yaxis_title_text="Repressor (uM)",  
                  zaxis_title_text="Activity")

fig.show()

In [ ]:
# Now repeat the same 3D graph as above but using matplotlib instead of plotly
fig = plt.figure(figsize=set_size(width, fraction=1, subplots=(2, 1)), dpi=300)
ax = fig.add_subplot(211,projection='3d')
ax2 = fig.add_subplot(212,projection='3d')

# Plot the surfaces
surf1 = ax.plot_surface(np.log10(AA), np.log10(RR), pLT, cmap=plt.cm.viridis, linewidth=0, antialiased=True, alpha=1.0)
surf2 = ax2.plot_surface(np.log10(AA), np.log10(RR), pLL, cmap=plt.cm.inferno, linewidth=0, antialiased=True, alpha=1.0)

# Add a colorbar for each surface and a label for them
fig.colorbar(surf1, ax=ax, shrink=0.5, aspect=10)
fig.colorbar(surf2, ax=ax2, shrink=0.5, aspect=10)

# Set the labels
ax.set_xlabel('$\log_{10}([A])$ ($\mu$M)')
ax.set_ylabel('$\log_{10}([R])$ ($\mu$M)')
ax.set_zlabel('Actividad (U.A.)')
ax.set_title('Respuesta del promotor pLux76Tet\nfrente a un gradiente de activadores y represores')

# Set the labels for the second plot
ax2.set_xlabel('$\log_{10}([A])$ ($\mu$M)')
ax2.set_ylabel('$\log_{10}([R])$ ($\mu$M)')
ax.set_zlabel('Actividad (U.A.)')
ax2.set_title('Respuesta del promotor pLas81Lac\nfrente a un gradiente de activadores y represores')

# Tight layout
plt.tight_layout()

# Change the view
ax.view_init(30, 45)
ax2.view_init(30, 45)

# Save the figure
plt.savefig('promoter_activity_thesis_SIMPLIFIED.pdf', dpi=600, bbox_inches='tight')